# MOVE ON HUR analysis notebook

This notebook loads a HUR isometric knee test file, corrects the output using the actual measured lever arm, and compares the results with reference values.

How to use this notebook:
1. **Run** the setup cell.
2. **Set** the input parameters.
3. **Select** a case from the dropdown menu.
4. **Run** the analysis cells.

In [ ]:
import urllib.request
UTILS_URL = "https://raw.githubusercontent.com/tuxotiikeri/MoveOn/main/src/hur_utils.py"
urllib.request.urlretrieve(UTILS_URL, "hur_utils.py")

from hur_utils import *
print("Utilities loaded.")

## 2) Select the casefile

Choose one example case from the dropdown menu.

In [ ]:
case_selector = create_case_dropdown(default="ACL")

## 3) Check the parameters for profile

Enter:
- the affected limb
- the reference group
- the actual measured lever arm in millimetres

Use the actual measured lever arm from the patient profile, not the approximate lever arm setting used in the HUR software.

In [ ]:
affected_limb = "left"     # options: "left", "right"
group = "athletic"         # options: "athletic", "adult", "elder"
actual_lever_mm = 330      # enter the actual measured lever arm in mm

## 4) Read the case file and show metadata

Make sure the defined lever arm matches the patient profile.

In [ ]:
FILEPATH, REFERENCE_CSV_PATH = download_case_and_reference(case_selector.value)

case = load_hur_case(FILEPATH)
show_case_metadata(case)
show_lever_arm_check(case, actual_lever_mm, affected_limb, group)

### Important note about lever arm correction

The HUR software uses an approximate lever arm setting.
This notebook allows the user to enter the actual measured lever arm in millimetres.

The corrected torque values are calculated by scaling the HUR software output using:

$$
\frac{\text{actual lever arm}}{\text{software lever arm}}
$$
  

This improves the accuracy of the reported peak torque values.

## 5) Peak torque results

The table below shows:
- `measured_kg` = raw sensor reading
- `actual_kg` = value corrected using the actual lever arm
- `torque_nm` = corrected peak torque
- `torque_nmkg` = corrected peak torque normalised to body mass


In [ ]:
peaks_corrected = build_corrected_peaks_from_case(case, actual_lever_mm)
show_peak_table(peaks_corrected)

## 6) The reference values

Reference values are read automatically from the tidy csv file.

Rules used here:
- **Extension** uses **mid-range** reference values
- **Flexion** uses **extended** reference values
- if the requested `athletic` or `elder` values are missing, the notebook falls back to the corresponding **adult** values

### Source
Sarabon et al. *Frontiers in Physiology* (2021), Table 1.  
Manual check of the source table: <https://pmc.ncbi.nlm.nih.gov/articles/instance/8554160/bin/Table_1.pdf>

In [ ]:
reference_selection = get_reference_values(group=group, gender=case["gender"])
show_reference_table(reference_selection)

## 7) Peak torque values with reference values

The plots visualize peak-torque and add:
- the **reference mean**
- the **95% reference interval**, showing the typical range in the reference population
- a short note on which reference values were used

In [ ]:
plot_peak_torque_with_reference(peaks_corrected, reference_selection)

## 8) H:Q ratio

The hamstring-to-quadriceps ratio is calculated within the same limb:

$$
H:Q = \frac{T_{\mathrm{flexion}}}{T_{\mathrm{extension}}}
$$

where \(T\) is the peak torque normalised to body mass Nm/kg

In [ ]:
calculate_hq_ratio(peaks_corrected)

## 9) Limb Symmetry Index (LSI)

The Limb Symmetry Index compares the affected limb to the contralateral limb:

$$
LSI(\%) = 100 \times \frac{T_{\mathrm{affected}}}{T_{\mathrm{contralateral}}}
$$

In [ ]:
calculate_lsi_table(peaks_corrected, affected_limb)

## 10) Data into long format

This conversion happens in the background.  
For teaching purposes, the notebook shows the first 10 rows of the converted table.

In [ ]:
ft_long = build_force_time_long_from_case(case, peaks_corrected)

In [ ]:
make_force_time_preview(ft_long, n=10)

## 11) Plot the torque-time curves

In [ ]:
plot_force_time_curves(ft_long)

## 12) Align the curves to contraction onset

Different curves may begin to rise at slightly different times.  
To compare the early phase fairly, the time axis is aligned to a common contraction onset.

### How the alignment was done
For each movement curve, contraction onset was defined as the first time point at which torque exceeded **5% of that curve's peak value** for at least **3 consecutive samples**.

Because the sampling interval is 20 ms, this means that the signal had to remain above the threshold for at least:

$$
3 \times 20\ \mathrm{ms} = 60\ \mathrm{ms}
$$

After the onset was detected, the time axis was shifted so that the onset occurred at:

$$
t = 0\ \mathrm{ms}
$$

In [ ]:
ft_aligned = align_to_onset(ft_long, value_col="torque_nm", pct=0.05, min_consecutive=3)
plot_force_time_curves(ft_aligned, aligned=True, title="Onset-aligned torque-time curves")

## 13) Calculate RFD100

This notebook reports only interval-based RFD values.  
Instantaneous RFD is intentionally left out here because the available sampling interval is 20 ms.

In [ ]:
rfd_summary = build_rfd_summary(ft_aligned)
show_rfd_summary(rfd_summary)

In [ ]:
plot_rfd_bars(rfd_summary, interval_ms=100)

## 14) Plot RFD200

In [ ]:
plot_rfd_bars(rfd_summary, interval_ms=200)

## 15) Final summary table

This table combines the main outcome variables used in interpretation.

In [ ]:
build_final_summary(peaks_corrected, rfd_summary)